# 07 基线分割对应MRI全序列核查

范围是当前54个病灶基线候选、41名患者、48组检查的掩膜对应序列，共6,936张DICOM；不是全部42,385张影像，也不包含本轮未恢复的全部FLAIR序列。先运行`restore_sources.py`，恢复已有MRI分卷后运行`restore_baseline_mask_series.py`。也可通过其`--download-missing`参数从原始公开Figshare恢复所需成员。

In [1]:
from pathlib import Path
import sys,json,runpy,gzip,hashlib
import pandas as pd
ROOT=Path.cwd()
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
manifest=ROOT/'data/metadata/baseline_qc/mask_series_members.jsonl.gz'
raw=gzip.decompress(manifest.read_bytes())
entries=[json.loads(x) for x in raw.decode().splitlines()]
assert len(entries)==6936
assert len({x['member'] for x in entries})==6936
print('Input manifest SHA256:',hashlib.sha256(raw).hexdigest())

Input manifest SHA256: 4265a67e6e34644153502b5c48d34c4ed521a20ac8689ff68348a03152f8b4c4


## 逐文件及逐序列检查

像素解码、SHA256、CRC32和头信息检查覆盖全部所选切片。按空间位置排序，以首末切片位置差除以间隔数构造多切片仿射，并对所有中间层检查残差；这避免相邻位移中位数的舍入误差累计。相邻层间距中位数仍用于报告和不规则间距诊断。工程容差保持不变，不代表临床阈值。

In [2]:
run=runpy.run_path(str(ROOT/'scripts/audit_additional_imaging.py'))['run']
summary=run(manifest_relative='data/metadata/baseline_qc/mask_series_members.jsonl',prefix='cohort',output_subdir='baseline-qc-20261028',scope='complete MRI series named by 54 baseline mask candidates; not all modalities',mask_candidates_only=True,write_overlays=False)
f=pd.read_csv(ROOT/'data/processed/cohort_dicom_file_qc.csv.gz')
s=pd.read_csv(ROOT/'data/processed/cohort_dicom_series_qc.csv')
assert len(f)==s.files.sum()==6936
assert f.sha256_match.sum()==f.crc32_match.sum()==6936
print(s[['patient_id','study_date_code','series_name','files','constant_slices','spacing_tag_position_mismatch','technical_stack_pass']].to_string(index=False))

{
  "scope": "complete MRI series named by 54 baseline mask candidates; not all modalities",
  "studies": 48,
  "dicom_files": 6936,
  "series_uid_groups": 48,
  "series_directories": 48,
  "integrity_verified_files": 6936,
  "pixel_decodable_files": 6936,
  "nonfinite_pixel_arrays": 0,
  "constant_pixel_arrays": 3,
  "date_mismatch_files": 0,
  "duplicate_sop_files": 0,
  "regular_geometry_stacks": 48,
  "technical_pass_stacks": 48,
  "t2_flair_metadata_candidates": 0,
  "t1_flair_names_not_t2_flair": 0,
  "derived_or_projection_groups": 7,
  "spacing_tag_position_mismatch_groups": 26,
  "t2_flair_candidates_with_interslice_gap": 0,
  "contrast_bolus_tag_populated_files": 0,
  "baseline_mask_candidates": 54,
  "grid_equivalent_masks": 42,
  "engineering_tolerances": {
    "direction_or_spacing_abs": 0.0001,
    "duplicate_plane_mm": 0.001,
    "plane_step_abs_mm": 0.05,
    "plane_step_relative": 0.01,
    "affine_residual_mm": 0.1,
    "grid_corner_voxels": 0.001
  },
  "manifest_sha

## 名称对应与网格误差

先保留精确名称匹配；只允许唯一的标点/大小写变体对应，歧义时停止，不进行模糊匹配。原掩膜与DICOM名称均保留，最终由空间坐标独立验证。全体积角点误差≤0.001体素且为完整有符号轴置换时才记录网格等价。超阈值记录不通过放宽容差、修改掩膜或插值转为通过。

In [3]:
m=pd.read_csv(ROOT/'data/processed/cohort_baseline_mask_alignment.csv')
assert len(m)==m.lesion_key.nunique()==54
print('Name mapping:',m.name_match_rule.value_counts().to_dict())
print('Alignment:',m.alignment_status.value_counts().to_dict())
print(m.loc[m.grid_equivalent==0,['lesion_key','max_corner_error_voxels','max_corner_error_mm','target_max_reindex_error_mm','alignment_status']].to_string(index=False))
assert m.anatomical_review_complete.sum()==m.use_for_model.sum()==0

Name mapping: {'exact': 43, 'unique_punctuation_case_variant': 11}
Alignment: {'grid_equivalent_no_interpolation': 42, 'grid_mismatch_requires_review': 12}
lesion_key  max_corner_error_voxels  max_corner_error_mm  target_max_reindex_error_mm              alignment_status
  20034_L1                 0.001223             0.001236                     0.001046 grid_mismatch_requires_review
  20034_L2                 0.001223             0.001236                     0.000608 grid_mismatch_requires_review
  40011_L2                 0.008304             0.005856                     0.003203 grid_mismatch_requires_review
  40074_L1                 0.001009             0.000743                     0.000417 grid_mismatch_requires_review
  40074_L2                 0.001009             0.000743                     0.000566 grid_mismatch_requires_review
  40084_L1                 0.005824             0.007623                     0.003051 grid_mismatch_requires_review
  40084_L2                 0.148

## 常量切片与目标体素

常量切片单独列为需要检查的项目，规则几何通过不等于图像质量通过。分割体素在连续DICOM坐标中的视野覆盖、以及最近层索引是否落在常量切片上，仅作为诊断性检查，不重采样或修改原标签。

In [4]:
print(f.loc[f.constant_pixel_array==1,['patient_id','study_date_code','InstanceNumber','pixel_min','pixel_max']].to_string(index=False))
print('Target voxels on constant slices:',m.target_voxels_on_constant_slices.sum())
print('All target voxels inside image FOV:',bool(m.target_voxels_inside_image_fov.eq(m.target_voxels).all()))
print('Full cohort clinical image review:',summary['clinical_image_review_complete'])

 patient_id  study_date_code  InstanceNumber  pixel_min  pixel_max
      40102         19000101             386          0          0
      40184         19000101             125          0          0
      40184         19000101             126          0          0
Target voxels on constant slices: 0
All target voxels inside image FOV: True
Full cohort clinical image review: False


## 研究状态

本轮不实施跨序列配准、不重画分割，也不把几何通过转换成临床终点通过。下一步调查仍超出网格容差的病灶，恢复和核实对应FLAIR，再建立配准方案。影像合格不能弥补125条未知结局和缺少已核实阴性对照的问题。

方法：[NiBabel多切片仿射](https://nipy.org/nibabel/dicom/dicom_orientation.html#getting-a-3d-affine-from-a-dicom-slice-or-list-of-slices)、[DICOM图像平面](https://dicom.nema.org/medical/dicom/2026c/output/chtml/part03/sect_C.7.6.2.html)。

In [5]:
c=pd.read_csv(ROOT/'data/processed/clinical_current.csv')
assert c.rn_time_day.isna().sum()==125
assert c.verified_non_event.sum()==0
assert summary['models_fitted'] is False
print('RN unknown: 125; verified non-events: 0; no model fitted.')

RN unknown: 125; verified non-events: 0; no model fitted.
